## Notebook overview

This notebook teaches k-means clustering using scikit-learn, applied to a banking-style dataset saved as `vs_bank.csv`.

You will:
- Load the dataset from a CSV file.
- Select four numeric variables for clustering:
  - `demog_pr`
  - `i_demog_age`
  - `ri_demog_homeval`
  - `ri_demog_inc`
- Handle missing values in a simple way.
- Standardise the variables (important for k-means).
- Fit k-means with 4 and 5 clusters.
- Compare results using within-cluster sum of squares (WCSS).
- Interpret clusters using simple summaries and plots.

---

## Dataset Instructions (Important)

The file `vs_bank.csv` must be downloaded from Moodle.

To use it in Google Colab:

1. Download `vs_bank.csv` from Moodle.
2. Open the notebook in Colab.
3. Click the folder icon on the left panel.
4. Click **Upload** and upload `vs_bank.csv` into the session.
5. Ensure the file name appears exactly as `vs_bank.csv`.

After uploading, the following line will work:

```python
df = pd.read_csv("vs_bank.csv")
```

Note that files uploaded to a Colab session are temporary and will need to be uploaded again if the session resets.

## Indentation and comments reminder

Python uses indentation (usually 4 spaces) to define blocks, such as inside `if`, `for`, `while`, and functions.  
Comments start with `#` and are ignored by Python. They are used to explain code.

In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt

# Scikit-learn tools
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


## Load the CSV file

This section loads `vs_bank.csv` into a DataFrame. If you are using Google Colab, upload the CSV into the session first.

In [ ]:
# Load the dataset
df = pd.read_csv("vs_bank.csv")

# Basic inspection
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nFirst 5 rows:\n", df.head())


## Select variables and basic checks

We will create a smaller DataFrame containing only the four variables needed for clustering.  
We also check missing values and basic summary statistics.

In [ ]:
# Select the variables for clustering
features = ["demog_pr", "i_demog_age", "ri_demog_homeval", "ri_demog_inc"]
X_raw = df[features].copy()

print("Selected feature shape:", X_raw.shape)
print("\nMissing values per column:\n", X_raw.isna().sum())
print("\nSummary statistics:\n", X_raw.describe())


## Handle missing values

k-means cannot handle missing values.

For this particular dataset (`vs_bank.csv`), the selected variables have already been imputed, so there are no missing values in these columns.

However, in real-world datasets this is often not the case.

The next cell keeps the code for median imputation as a general template. You can reuse that pattern when working with other datasets that contain missing values.

In [ ]:
# Fill missing values with the median of each column (template for other datasets)
X_filled = X_raw.copy()

for col in features:
    X_filled[col] = X_filled[col].fillna(X_filled[col].median())

print("Missing values after filling:\n", X_filled.isna().sum())
print("\nPreview:\n", X_filled.head())


## Standardise the variables

k-means is distance-based. If variables are on different scales (for example income versus age), the larger-scale variables can dominate the clustering.

Standardisation converts each feature to have mean 0 and standard deviation 1.

In [ ]:
# Standardise features
scaler = StandardScaler()
X = scaler.fit_transform(X_filled)

print("Scaled array shape:", X.shape)
print("First row (scaled):", X[0])


## Choosing the number of clusters using WCSS

We will compute **within-cluster sum of squares (WCSS)** for different values of k.

- WCSS measures how close points in each cluster are to their cluster centre.
- Lower WCSS means tighter clusters, but WCSS always decreases as k increases.
- The elbow plot helps find a k where improvements start to diminish.

In [ ]:
ks = range(2, 11)
wcss = []

for k in ks:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    kmeans.fit(X)

    # WCSS is available as the sum of squared distances to cluster centres
    # scikit-learn stores this value in the attribute called inertia_
    wcss.append(kmeans.inertia_)

print("k values:", list(ks))
print("WCSS:", [round(v, 2) for v in wcss])

# Elbow plot (WCSS vs k)
plt.figure()
plt.plot(list(ks), wcss, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Within-cluster sum of squares (WCSS)")
plt.title("Elbow Plot (WCSS vs k)")
plt.show()


## Fit k-means for k = 4 and k = 5

Now we will train two k-means models: one with 4 clusters and one with 5 clusters.  
We will store cluster labels in a DataFrame for profiling and interpretation.

In [ ]:
# Fit k-means with 4 clusters
kmeans_4 = KMeans(n_clusters=4, n_init=10, random_state=42)
labels_4 = kmeans_4.fit_predict(X)

# Fit k-means with 5 clusters
kmeans_5 = KMeans(n_clusters=5, n_init=10, random_state=42)
labels_5 = kmeans_5.fit_predict(X)

# Attach cluster labels to a copy of the filled data for profiling
df_clusters = X_filled.copy()
df_clusters["cluster_4"] = labels_4
df_clusters["cluster_5"] = labels_5

print(df_clusters.head())


## Compare k = 4 versus k = 5 using WCSS

We will compare WCSS for the two models.

- Lower WCSS usually means tighter clusters.
- In practice, you also consider interpretability and usefulness for the business context.

In [ ]:
wcss_4 = kmeans_4.inertia_
wcss_5 = kmeans_5.inertia_

print("WCSS for k=4:", round(wcss_4, 2))
print("WCSS for k=5:", round(wcss_5, 2))
print("\nNote: WCSS will usually be smaller for larger k. Use the elbow plot and interpretability to choose k.")


## Cluster profiling

Cluster labels become useful when you can describe each cluster in business terms.

We will compute:
- cluster sizes
- mean and median of each variable per cluster

In [ ]:
# Cluster sizes (k=4)
print("Cluster sizes (k=4):\n", df_clusters["cluster_4"].value_counts().sort_index())

# Mean profile (k=4)
print("\nMean profile (k=4):\n", df_clusters.groupby("cluster_4")[features].mean())

# Median profile (k=4)
print("\nMedian profile (k=4):\n", df_clusters.groupby("cluster_4")[features].median())


## Optional: Visualise clusters in two dimensions

A simple way to visualise clustering is to plot two variables at a time and colour points by cluster.  
This is not a complete view (because clustering used four dimensions), but it helps build intuition.

In [ ]:
# Scatter plot: income vs age
plt.figure()
for c in sorted(df_clusters["cluster_4"].unique()):
    subset = df_clusters[df_clusters["cluster_4"] == c]
    plt.scatter(subset["i_demog_age"], subset["ri_demog_inc"], label=f"Cluster {c}", alpha=0.7)

plt.xlabel("Age (i_demog_age)")
plt.ylabel("Income (ri_demog_inc)")
plt.title("K-means Clusters (k=4): Age vs Income")
plt.legend()
plt.show()

# Scatter plot: home value vs income
plt.figure()
for c in sorted(df_clusters["cluster_4"].unique()):
    subset = df_clusters[df_clusters["cluster_4"] == c]
    plt.scatter(subset["ri_demog_homeval"], subset["ri_demog_inc"], label=f"Cluster {c}", alpha=0.7)

plt.xlabel("Home value (ri_demog_homeval)")
plt.ylabel("Income (ri_demog_inc)")
plt.title("K-means Clusters (k=4): Home Value vs Income")
plt.legend()
plt.show()


## Next step: extending to categorical variables with k-prototypes

k-means works only with numeric variables. If your dataset also contains **categorical** variables (for example gender, suburb, product type), you can use **k-prototypes**, which is designed for mixed numeric + categorical data.

General steps:
1. Choose numeric columns and categorical columns.
2. Convert categorical columns to `object` or `category` type.
3. Fit `KPrototypes` and get cluster labels.
4. Compare different values of k using the cost reported by the model (similar idea to WCSS).

In [ ]:
# OPTIONAL EXTENSION (Mixed data): k-prototypes for numeric + categorical variables
# This cell is intentionally almost-complete. Students should fill the marked parts.

# If running in Colab, you may need to install kmodes first:
# !pip -q install kmodes

from kmodes.kprototypes import KPrototypes

# 1) Choose categorical columns (examples). Replace with real column names from vs_bank.csv
categorical_cols = [
    # "gender",
    # "marital_status",
    # "suburb"
]

# 2) Numeric columns (already used for k-means)
numeric_cols = ["demog_pr", "i_demog_age", "ri_demog_homeval", "ri_demog_inc"]

# 3) Build a mixed DataFrame (numeric + categorical)
#    Uncomment and adjust once you have real categorical columns.
# X_mix = df[numeric_cols + categorical_cols].copy()

# 4) Basic missing value handling (simple baseline)
#    For numeric: fill with median; for categorical: fill with a placeholder string.
# for col in numeric_cols:
#     X_mix[col] = X_mix[col].fillna(X_mix[col].median())
# for col in categorical_cols:
#     X_mix[col] = X_mix[col].fillna("Unknown")

# 5) k-prototypes expects a NumPy array; categorical columns are identified by their index positions.
# X_mix_np = X_mix.to_numpy()

# Indices of categorical columns in X_mix (they come after numeric columns if you stacked as numeric + categorical)
# cat_idx = list(range(len(numeric_cols), len(numeric_cols) + len(categorical_cols)))

# 6) Fit k-prototypes
# Choose k = 4 or 5 to match the earlier exercise
# kproto = KPrototypes(n_clusters=4, init="Cao", n_init=5, verbose=0, random_state=42)
# clusters = kproto.fit_predict(X_mix_np, categorical=cat_idx)

# 7) Attach cluster labels and profile results
# df_kproto = X_mix.copy()
# df_kproto["cluster_kproto"] = clusters

# print("Cluster sizes:\n", df_kproto["cluster_kproto"].value_counts().sort_index())
# print("\nNumeric means by cluster:\n", df_kproto.groupby("cluster_kproto")[numeric_cols].mean())

# OPTIONAL: see category distributions per cluster (replace 'gender' with a real column)
# print(pd.crosstab(df_kproto["cluster_kproto"], df_kproto["gender"], normalize="index"))

# Note:
# - k-prototypes has a 'cost_' attribute after fitting. Lower is better, but it typically decreases as k increases.
# - You can make an elbow plot of cost_ across k values, similar to the WCSS plot earlier.
